# Activity 7 — Linear Regression: Predicting Systolic Blood Pressure
## MRHA CardioWatch Program

**Course:** Introduction to Artificial Intelligence  
**Sessions:** 20 and 21  
**Libraries:** NumPy · Pandas · Matplotlib · scikit-learn · ipywidgets

---

## Background

Hypertension (high blood pressure) affects roughly **1.28 billion adults** worldwide and is a leading risk factor for heart disease and stroke. Predicting a patient's blood pressure from measurable characteristics allows physicians to identify at-risk individuals before complications arise.

The **Maplewood Regional Health Authority (MRHA)** runs the *CardioWatch* program, which monitors patients aged 25–75 who are at risk of developing hypertension. You are part of a data science team that has been given **200 anonymized patient records**. Your goal is to build regression models that predict each patient's **Systolic Blood Pressure (SBP)** — the top number in a blood pressure reading (e.g., the **128** in 128/82 mmHg).

**Clinical SBP reference ranges:**

| Category | SBP (mmHg) |
|----------|------------|
| Normal | < 120 |
| Elevated | 120 – 129 |
| High — Stage 1 | 130 – 139 |
| High — Stage 2 | ≥ 140 |

---

## Dataset

| Column | Description | Units |
|--------|-------------|-------|
| `age` | Patient age | years |
| `bmi` | Body Mass Index | kg/m² |
| `sbp` | Systolic Blood Pressure (**target**) | mmHg |

---

## Assignment Structure

| Task | Model | Predictor(s) |
|------|-------|--------------|
| **Task 1** | Simple Linear Regression | Age only |
| **Task 2** | Multiple Linear Regression | Age + BMI |

Each task follows the same pipeline:
1. Exploratory Data Analysis  
2. Train / Test Split  
3. Build and Train the Model  
4. Evaluate the Model  
5. Visualize Results  
6. Residual Analysis  
7. Interactive Experiment with ipywidgets

---
## Setup — Import Libraries

Run this cell first. It imports every library used throughout the notebook.

- **NumPy** — numerical arrays and math operations  
- **Pandas** — tabular data (DataFrames)  
- **Matplotlib** — plots and visualizations  
- **scikit-learn** — machine learning models and metrics  
- **ipywidgets** — interactive sliders inside the notebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import sklearn

import ipywidgets as widgets
from ipywidgets import interact

import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=3)
pd.options.display.float_format = '{:.2f}'.format

print('Libraries loaded successfully.')
print(f'  numpy      : {np.__version__}')
print(f'  pandas     : {pd.__version__}')
print(f'  sklearn    : {sklearn.__version__}')
print(f'  ipywidgets : {widgets.__version__}')

---
## Generate the MRHA Dataset

The cell below generates the synthetic patient dataset. The data follows a clinically realistic formula:

$$SBP = 105 + 0.55 \times Age + 1.10 \times BMI + \varepsilon$$

Where $\varepsilon$ is random Gaussian noise with $\sigma = 6$ mmHg, representing natural biological variability.

> **Do not modify this cell.** The fixed random seed (`7`) ensures every student works with the same 200 patients.

In [ ]:
# -------------------------------------------------------
# MRHA CardioWatch Dataset  —  DO NOT MODIFY
# -------------------------------------------------------
np.random.seed(7)
N = 200

age = np.random.uniform(25, 75, N)        # Age: 25 to 75 years
bmi = np.random.uniform(18.5, 38.0, N)   # BMI: 18.5 to 38 kg/m2

# True relationship (what the model should learn):
#   baseline ~105 mmHg
#   +0.55 mmHg per year of age
#   +1.10 mmHg per unit of BMI
#   + Gaussian noise (biological variability)
sbp = 105 + 0.55 * age + 1.10 * bmi + np.random.normal(0, 6, N)
sbp = np.clip(sbp, 95, 185)   # clip to physiologically plausible range

df = pd.DataFrame({
    'age': np.round(age, 1),
    'bmi': np.round(bmi, 1),
    'sbp': np.round(sbp, 1)
})

print(f'Dataset: {df.shape[0]} patients x {df.shape[1]} variables')
print()
print('First 8 rows:')
display(df.head(8))
print()
print('Summary statistics:')
display(df.describe())

---
---
# TASK 1 — Simple Linear Regression: SBP ~ Age

In this task we build a model that predicts Systolic Blood Pressure from **age alone**.

The model has the form:

$$\hat{SBP} = \beta_0 + \beta_1 \times Age$$

- $\beta_0$ — **intercept**: the predicted SBP when age = 0 (a mathematical baseline, not directly clinically meaningful)  
- $\beta_1$ — **slope**: how many mmHg SBP changes for each additional year of age  

The algorithm finds the values of $\beta_0$ and $\beta_1$ that minimize the **Sum of Squared Residuals (SSR)**:

$$SSR = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

This is called the **Ordinary Least Squares (OLS)** method.

---
### Step 1.1 — Exploratory Data Analysis

Before building any model, we must **understand the data**. We will:

1. Plot the distribution of the predictor (`age`) to see if it is roughly uniform
2. Create a scatter plot of `age` vs `sbp` to visually assess whether a linear relationship exists
3. Compute the Pearson correlation coefficient to quantify the linear relationship

**Pearson correlation (r)** measures the strength and direction of a linear relationship:  
- r close to +1 → strong positive linear relationship  
- r close to 0 → no linear relationship  
- r close to -1 → strong negative linear relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Plot 1: Distribution of Age ---
mean_age = df['age'].mean()
axes[0].hist(df['age'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(mean_age, color='firebrick', linestyle='--', linewidth=2,
                label=f'Mean = {mean_age:.1f} yrs')
axes[0].set_xlabel('Age (years)', fontsize=11)
axes[0].set_ylabel('Number of Patients', fontsize=11)
axes[0].set_title('Distribution of Patient Age', fontsize=12)
axes[0].legend(fontsize=9)

# --- Plot 2: Scatter plot Age vs SBP ---
axes[1].scatter(df['age'], df['sbp'], alpha=0.5, color='steelblue',
                edgecolors='white', linewidth=0.4, s=40, label='Patients')
# Clinical threshold reference lines
axes[1].axhline(120, color='gold',       linestyle='--', linewidth=1.3, label='Elevated (120)')
axes[1].axhline(130, color='darkorange', linestyle='--', linewidth=1.3, label='Stage 1 (130)')
axes[1].axhline(140, color='firebrick',  linestyle='--', linewidth=1.3, label='Stage 2 (140)')
axes[1].set_xlabel('Age (years)', fontsize=11)
axes[1].set_ylabel('Systolic Blood Pressure (mmHg)', fontsize=11)
axes[1].set_title('Age vs. Systolic Blood Pressure', fontsize=12)
axes[1].legend(fontsize=8)

plt.suptitle('Task 1 — Exploratory Data Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

r = df['age'].corr(df['sbp'])
print(f'Pearson correlation  (Age vs SBP) : r = {r:.3f}')
print()
print('Interpretation:')
if abs(r) > 0.7:
    strength = 'strong'
elif abs(r) > 0.4:
    strength = 'moderate'
else:
    strength = 'weak'
direction = 'positive' if r > 0 else 'negative'
print(f'  r = {r:.3f} suggests a {strength} {direction} linear relationship.')
print('  As age increases, SBP tends to increase — consistent with clinical knowledge.')

---
### Step 1.2 — Train / Test Split

We split the data into two subsets:

- **Training set (80%)** — used to fit the model (learn $\beta_0$ and $\beta_1$)
- **Test set (20%)** — held out and used only to evaluate the model on **unseen data**

Why split? If we evaluate on the same data we trained on, the model could **memorize** the training examples and report artificially good performance. The test set simulates how the model performs on new patients the clinic has never seen before.

The `random_state` parameter controls how the data is shuffled before splitting. Using the same value guarantees the same split every time — important for reproducibility.

> **Note on shape:** scikit-learn's `LinearRegression` expects the feature matrix `X` to be 2-dimensional: shape `(n_samples, n_features)`. Even for one feature we must reshape from `(200,)` to `(200, 1)` using `.values.reshape(-1, 1)`.

In [ ]:
# Feature matrix: age only — shape must be (n_samples, 1) for scikit-learn
X1 = df[['age']].values          # shape (200, 1)
y  = df['sbp'].values             # shape (200,)

# Split: 80% train, 20% test
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y, test_size=0.20, random_state=7
)

print('Train / Test Split Summary')
print(f'  Total patients    : {len(y)}')
print(f'  Training samples  : {len(y1_train)}  ({len(y1_train)/len(y):.0%})')
print(f'  Test samples      : {len(y1_test)}   ({len(y1_test)/len(y):.0%})')
print(f'  X1_train shape    : {X1_train.shape}')
print(f'  X1_test  shape    : {X1_test.shape}')

---
### Step 1.3 — Build and Train the Model

We use scikit-learn's `LinearRegression` class. Under the hood, it applies the **Ordinary Least Squares** closed-form solution:

$$\hat{\boldsymbol{\beta}} = (X^T X)^{-1} X^T y$$

This formula computes the coefficients that minimize the sum of squared residuals in one shot — no iterative training needed for linear regression.

After fitting, the learned coefficients are stored in:
- `model.intercept_` → $\beta_0$  
- `model.coef_[0]`   → $\beta_1$

In [ ]:
# Create the model object
model1 = LinearRegression()

# Train on the training set
# .fit() computes the OLS solution and stores the coefficients
model1.fit(X1_train, y1_train)

b0_1 = model1.intercept_   # intercept
b1_1 = model1.coef_[0]     # slope for age

print('=== Simple Linear Regression — Learned Coefficients ===')
print(f'  Intercept  b0 : {b0_1:.4f} mmHg')
print(f'  Slope      b1 : {b1_1:.4f} mmHg per year')
print()
print(f'  Equation: SBP = {b0_1:.2f} + {b1_1:.2f} x Age')
print()
print('Clinical interpretation of the slope:')
print(f'  For each additional year of age, SBP increases by {b1_1:.2f} mmHg on average,')
print('  holding all other factors constant.')

---
### Step 1.4 — Evaluate the Model

We evaluate the model on the **test set** using two standard metrics:

#### Mean Squared Error (MSE)
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

- Measures the average **squared** difference between actual and predicted values
- Lower is better
- Units: mmHg² (taking the square root gives RMSE in mmHg — easier to interpret)

#### Coefficient of Determination (R²)
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2} = 1 - \frac{SSE}{SST}$$

- Measures the **proportion of variance** in SBP explained by the model
- Ranges from 0 to 1 (higher is better)
- R² = 0.70 means the model explains 70% of the variability in SBP
- R² = 1.00 would be a perfect model

In [ ]:
# Generate predictions on the test set
y1_pred = model1.predict(X1_test)

# Compute metrics
mse1  = mean_squared_error(y1_test, y1_pred)
rmse1 = np.sqrt(mse1)
r2_1  = r2_score(y1_test, y1_pred)

print('=== Task 1 — Model Evaluation on Test Set ===')
print(f'  Mean Squared Error  (MSE)  : {mse1:.3f} mmHg2')
print(f'  Root MSE            (RMSE) : {rmse1:.3f} mmHg')
print(f'  R-squared           (R2)   : {r2_1:.3f}')
print()
print(f'  Interpretation: the model explains {r2_1*100:.1f}% of the variance in SBP.')
print(f'  On average, predictions are off by about {rmse1:.1f} mmHg.')

---
### Step 1.5 — Visualize the Regression Line

The plot below shows:
- **Blue dots** — actual test patients
- **Gray dots** — training patients (for reference)
- **Red line** — the regression line fitted by the model
- **Dashed horizontal lines** — clinical blood pressure thresholds

A good regression line should run through the center of the point cloud. Patients far from the line represent large prediction errors (residuals).

In [ ]:
# Generate a smooth line across the full age range for plotting
age_range = np.linspace(24, 76, 300).reshape(-1, 1)
sbp_line1 = model1.predict(age_range)

fig, ax = plt.subplots(figsize=(10, 6))

# Training data (background, low opacity)
ax.scatter(X1_train, y1_train, alpha=0.18, color='gray',
           edgecolors='none', s=30, label='Training patients')

# Test data
ax.scatter(X1_test, y1_test, alpha=0.65, color='steelblue',
           edgecolors='white', linewidth=0.4, s=45, label='Test patients', zorder=3)

# Regression line
ax.plot(age_range, sbp_line1, color='firebrick', linewidth=2.5, zorder=4,
        label=f'SBP = {b0_1:.1f} + {b1_1:.2f} x Age   (R2={r2_1:.3f})')

# Clinical thresholds
ax.axhline(120, color='gold',       linestyle='--', linewidth=1.2, alpha=0.85, label='Elevated (120 mmHg)')
ax.axhline(130, color='darkorange', linestyle='--', linewidth=1.2, alpha=0.85, label='Stage 1  (130 mmHg)')
ax.axhline(140, color='red',        linestyle='--', linewidth=1.2, alpha=0.85, label='Stage 2  (140 mmHg)')

ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Systolic Blood Pressure (mmHg)', fontsize=12)
ax.set_title('Task 1 — Simple Linear Regression: SBP vs. Age', fontsize=13)
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(24, 76)
ax.set_ylim(88, 192)

plt.tight_layout()
plt.show()

---
### Step 1.6 — Residual Analysis

A **residual** is the difference between the actual value and the model's prediction:

$$e_i = y_i - \hat{y}_i$$

Residual analysis lets us check whether the **assumptions of linear regression** are satisfied:

| Assumption | What to look for in the plots |
|------------|------------------------------|
| **Linearity** | Residuals scattered randomly around zero — no curved pattern |
| **Homoscedasticity** | Residual spread is constant across all predicted values |
| **Normality of errors** | Residual histogram is approximately bell-shaped |
| **Independence** | No systematic patterns across the x-axis |

If these assumptions hold, OLS gives the **Best Linear Unbiased Estimator (BLUE)**.

In [ ]:
residuals1 = y1_test - y1_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Plot 1: Residuals vs. Fitted Values ---
axes[0].scatter(y1_pred, residuals1, alpha=0.6, color='steelblue',
                edgecolors='white', linewidth=0.3, s=40)
axes[0].axhline(0, color='firebrick', linewidth=2, linestyle='--', label='Zero residual line')
axes[0].set_xlabel('Predicted SBP (mmHg)', fontsize=11)
axes[0].set_ylabel('Residual (mmHg)', fontsize=11)
axes[0].set_title('Residuals vs. Fitted Values', fontsize=12)
axes[0].legend(fontsize=8)
# Add a subtle horizontal band around zero to guide the eye
axes[0].axhspan(-rmse1, rmse1, alpha=0.07, color='steelblue', label='+/- RMSE band')

# --- Plot 2: Histogram of Residuals ---
mean_resid = residuals1.mean()
axes[1].hist(residuals1, bins=18, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0,           color='firebrick', linewidth=2,   linestyle='--', label='Zero')
axes[1].axvline(mean_resid,  color='gold',      linewidth=1.5, linestyle='-',
                label=f'Mean residual = {mean_resid:.2f}')
axes[1].set_xlabel('Residual (mmHg)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title(f'Distribution of Residuals  (RMSE = {rmse1:.2f} mmHg)', fontsize=12)
axes[1].legend(fontsize=8)

plt.suptitle('Task 1 — Residual Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Residual summary:')
print(f'  Mean   : {residuals1.mean():.4f}  (should be close to 0)')
print(f'  Std    : {residuals1.std():.4f} mmHg')
print(f'  Min    : {residuals1.min():.2f} mmHg')
print(f'  Max    : {residuals1.max():.2f} mmHg')

---
### Step 1.7 — Interactive Experiment with ipywidgets

Use the sliders below to explore how the model behaves under different conditions:

| Slider | What it controls | What to observe |
|--------|------------------|-----------------|
| **Test Size** | Fraction of data held out for testing | Smaller test sets → noisier metric estimates |
| **Noise Level (σ)** | Standard deviation of biological variability in the data | More noise → lower R², higher MSE |
| **Random Seed** | Which patients go to train vs. test | Shows how metrics vary across different splits |

**Try this:** Set noise to 2 and observe R². Then raise it to 20. What happens to the regression line and R²?

In [ ]:
def run_simple_lr(test_size, noise_std, random_seed):
    # Regenerate data with the chosen noise level
    np.random.seed(random_seed)
    n = 200
    age_w = np.random.uniform(25, 75, n)
    sbp_w = 105 + 0.55 * age_w + np.random.normal(0, noise_std, n)
    sbp_w = np.clip(sbp_w, 95, 185)

    X_w = age_w.reshape(-1, 1)
    y_w = sbp_w

    # Split
    Xtr, Xte, ytr, yte = train_test_split(
        X_w, y_w, test_size=test_size, random_state=random_seed
    )

    # Train
    m = LinearRegression()
    m.fit(Xtr, ytr)
    ypred = m.predict(Xte)

    mse_w  = mean_squared_error(yte, ypred)
    rmse_w = np.sqrt(mse_w)
    r2_w   = r2_score(yte, ypred)
    resid  = yte - ypred
    b0_w   = m.intercept_
    b1_w   = m.coef_[0]

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Simple LR  |  test_size={test_size:.0%}  noise={noise_std} mmHg  seed={random_seed}',
        fontsize=12
    )

    age_line = np.linspace(24, 76, 300).reshape(-1, 1)
    sbp_line = m.predict(age_line)

    # Subplot 1: regression line
    axes[0].scatter(Xtr, ytr, alpha=0.15, color='gray', edgecolors='none', s=25, label='Train')
    axes[0].scatter(Xte, yte, alpha=0.6,  color='steelblue', edgecolors='white',
                    linewidth=0.3, s=38, label='Test', zorder=3)
    axes[0].plot(age_line, sbp_line, color='firebrick', linewidth=2.3, zorder=4,
                 label=f'SBP = {b0_w:.1f} + {b1_w:.2f}*Age')
    axes[0].axhline(120, color='gold',       linestyle='--', linewidth=1, alpha=0.7)
    axes[0].axhline(130, color='darkorange', linestyle='--', linewidth=1, alpha=0.7)
    axes[0].axhline(140, color='red',        linestyle='--', linewidth=1, alpha=0.7)
    axes[0].set_xlabel('Age (years)', fontsize=11)
    axes[0].set_ylabel('SBP (mmHg)', fontsize=11)
    axes[0].set_title(f'Regression Line  |  R2 = {r2_w:.3f}  MSE = {mse_w:.1f}', fontsize=11)
    axes[0].legend(fontsize=8)
    axes[0].set_xlim(24, 76)
    axes[0].set_ylim(85, 195)

    # Subplot 2: residual histogram
    axes[1].hist(resid, bins=18, color='steelblue', edgecolor='white', alpha=0.85)
    axes[1].axvline(0,           color='firebrick', linewidth=2,   linestyle='--')
    axes[1].axvline(resid.mean(),color='gold',      linewidth=1.5, linestyle='-',
                    label=f'Mean = {resid.mean():.2f}')
    axes[1].set_xlabel('Residual (mmHg)', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_title(f'Residuals  |  RMSE = {rmse_w:.2f} mmHg', fontsize=11)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f'  Train: {len(ytr)} patients  |  Test: {len(yte)} patients')
    print(f'  Coefficients : SBP = {b0_w:.2f} + {b1_w:.3f} * Age')
    print(f'  MSE={mse_w:.3f}  RMSE={rmse_w:.3f} mmHg  R2={r2_w:.3f}')


widgets.interact(
    run_simple_lr,
    test_size=widgets.FloatSlider(
        min=0.10, max=0.40, step=0.05, value=0.20,
        description='Test Size:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    ),
    noise_std=widgets.FloatSlider(
        min=2.0, max=25.0, step=1.0, value=6.0,
        description='Noise Level (sigma):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    ),
    random_seed=widgets.IntSlider(
        min=0, max=50, step=1, value=7,
        description='Random Seed:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    )
)

---
---
# TASK 2 — Multiple Linear Regression: SBP ~ Age + BMI

A patient's blood pressure is influenced by more than just age. **Body Mass Index (BMI)** is another well-established risk factor — excess body weight increases cardiovascular strain and raises blood pressure.

In this task we extend the model to use **two predictors simultaneously**:

$$\hat{SBP} = \beta_0 + \beta_1 \times Age + \beta_2 \times BMI$$

- $\beta_1$ — change in SBP per year of age, **holding BMI constant**
- $\beta_2$ — change in SBP per unit of BMI, **holding age constant**

The key insight is that each coefficient represents the **isolated effect** of one variable after removing the contribution of the other. This is the power of multiple regression over running two separate simple regressions.

Geometrically, the model now fits a **plane** in 3D space (Age, BMI, SBP) instead of a line.

---
### Step 2.1 — Exploratory Data Analysis (Two Variables)

We now look at:
1. The distribution of BMI
2. BMI vs SBP scatter plot
3. The correlation between both predictors and the target
4. The correlation between the two predictors themselves (multicollinearity check)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# --- Top-left: Distribution of BMI ---
mean_bmi = df['bmi'].mean()
axes[0, 0].hist(df['bmi'], bins=20, color='darkorange', edgecolor='white', alpha=0.85)
axes[0, 0].axvline(mean_bmi, color='firebrick', linestyle='--', linewidth=2,
                   label=f'Mean = {mean_bmi:.1f} kg/m2')
axes[0, 0].axvline(25, color='gold',   linestyle=':', linewidth=1.5, label='Overweight (25)')
axes[0, 0].axvline(30, color='red',    linestyle=':', linewidth=1.5, label='Obese (30)')
axes[0, 0].set_xlabel('BMI (kg/m2)', fontsize=11)
axes[0, 0].set_ylabel('Number of Patients', fontsize=11)
axes[0, 0].set_title('Distribution of BMI', fontsize=12)
axes[0, 0].legend(fontsize=8)

# --- Top-right: BMI vs SBP ---
axes[0, 1].scatter(df['bmi'], df['sbp'], alpha=0.5, color='darkorange',
                   edgecolors='white', linewidth=0.4, s=40)
axes[0, 1].axhline(120, color='gold',       linestyle='--', linewidth=1.2, label='Elevated (120)')
axes[0, 1].axhline(130, color='darkorange', linestyle='--', linewidth=1.2, label='Stage 1 (130)')
axes[0, 1].axhline(140, color='firebrick',  linestyle='--', linewidth=1.2, label='Stage 2 (140)')
axes[0, 1].set_xlabel('BMI (kg/m2)', fontsize=11)
axes[0, 1].set_ylabel('Systolic Blood Pressure (mmHg)', fontsize=11)
axes[0, 1].set_title('BMI vs. Systolic Blood Pressure', fontsize=12)
axes[0, 1].legend(fontsize=8)

# --- Bottom-left: Age vs BMI (check for multicollinearity) ---
axes[1, 0].scatter(df['age'], df['bmi'], alpha=0.45, color='mediumpurple',
                   edgecolors='white', linewidth=0.4, s=40)
r_ab = df['age'].corr(df['bmi'])
axes[1, 0].set_xlabel('Age (years)', fontsize=11)
axes[1, 0].set_ylabel('BMI (kg/m2)', fontsize=11)
axes[1, 0].set_title(f'Age vs. BMI  (r = {r_ab:.3f})', fontsize=12)
axes[1, 0].text(0.05, 0.92, f'r = {r_ab:.3f}', transform=axes[1, 0].transAxes,
                fontsize=10, color='firebrick')

# --- Bottom-right: Full correlation matrix as text ---
corr = df.corr()
im = axes[1, 1].imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1, 1].set_xticks(range(len(corr)))
axes[1, 1].set_yticks(range(len(corr)))
axes[1, 1].set_xticklabels(corr.columns, fontsize=11)
axes[1, 1].set_yticklabels(corr.columns, fontsize=11)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[1, 1].text(j, i, f'{corr.values[i, j]:.2f}',
                        ha='center', va='center', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1, 1])
axes[1, 1].set_title('Correlation Matrix', fontsize=12)

plt.suptitle('Task 2 — Exploratory Data Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

r_bs = df['bmi'].corr(df['sbp'])
r_as = df['age'].corr(df['sbp'])
print(f'Pearson correlation  Age vs SBP : r = {r_as:.3f}')
print(f'Pearson correlation  BMI vs SBP : r = {r_bs:.3f}')
print(f'Pearson correlation  Age vs BMI : r = {r_ab:.3f}  (multicollinearity check)')
print()
print('Note: if Age and BMI were highly correlated with each other (|r| > 0.8),')
print('it would cause multicollinearity problems in multiple regression.')
print(f'Here |r| = {abs(r_ab):.3f}, which is acceptable.')

---
### Step 2.2 — Prepare the Feature Matrix

For multiple regression, `X` is a **matrix** with one column per feature. scikit-learn handles this naturally — the same `LinearRegression` class works for any number of predictors.

We reuse the same `random_state=7` and `test_size=0.20` as Task 1 so both models are evaluated on the **same test patients**, making the comparison fair.

In [ ]:
# Feature matrix with TWO columns: age and bmi
X2 = df[['age', 'bmi']].values   # shape (200, 2)
y  = df['sbp'].values             # same target as Task 1

# Same split parameters as Task 1 — ensures same test patients
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.20, random_state=7
)

print('Feature matrix shapes:')
print(f'  X2       : {X2.shape}  (200 patients, 2 features)')
print(f'  X2_train : {X2_train.shape}')
print(f'  X2_test  : {X2_test.shape}')
print()
print('First 5 rows of X2_train (columns = [age, bmi]):')
print(X2_train[:5])

---
### Step 2.3 — Build and Train the Multiple Regression Model

The OLS formula extends naturally to multiple predictors:

$$\hat{\boldsymbol{\beta}} = (X^T X)^{-1} X^T y$$

where $X$ is now an $n \times 3$ matrix (intercept column + age column + BMI column) and $\boldsymbol{\beta} = [\beta_0, \beta_1, \beta_2]^T$.

The scikit-learn API is identical to Task 1 — only the shape of `X` changes.

In [ ]:
# Create and train the multiple regression model
model2 = LinearRegression()
model2.fit(X2_train, y2_train)

b0_2 = model2.intercept_
b1_2 = model2.coef_[0]   # age coefficient
b2_2 = model2.coef_[1]   # bmi coefficient

print('=== Multiple Linear Regression — Learned Coefficients ===')
print(f'  Intercept  b0 : {b0_2:.4f} mmHg')
print(f'  Age coef.  b1 : {b1_2:.4f} mmHg per year')
print(f'  BMI coef.  b2 : {b2_2:.4f} mmHg per kg/m2')
print()
print(f'  Equation: SBP = {b0_2:.2f} + {b1_2:.2f}*Age + {b2_2:.2f}*BMI')
print()
print('Clinical interpretation:')
print(f'  - Each additional year of age raises SBP by {b1_2:.2f} mmHg (holding BMI constant).')
print(f'  - Each additional unit of BMI raises SBP by {b2_2:.2f} mmHg (holding age constant).')

---
### Step 2.4 — Evaluate and Compare Both Models

Now we compute the same metrics as Task 1 and place them side by side. The key question is: **how much does adding BMI improve the model?**

The improvement in R² tells us the additional percentage of SBP variance that BMI explains beyond age alone.

In [ ]:
# Predict on the test set
y2_pred = model2.predict(X2_test)

# Metrics
mse2  = mean_squared_error(y2_test, y2_pred)
rmse2 = np.sqrt(mse2)
r2_2  = r2_score(y2_test, y2_pred)

# Note: y1_test and y2_test are the same patients because we used random_state=7
# We compare against y1_pred from Task 1
print('=' * 55)
print(f'{"Metric":<30} {"Simple LR":>10} {"Multiple LR":>12}')
print('-' * 55)
print(f'{"Predictors":<30} {"Age only":>10} {"Age + BMI":>12}')
print(f'{"MSE (mmHg2)":<30} {mse1:>10.3f} {mse2:>12.3f}')
print(f'{"RMSE (mmHg)":<30} {rmse1:>10.3f} {rmse2:>12.3f}')
print(f'{"R-squared":<30} {r2_1:>10.3f} {r2_2:>12.3f}')
print('=' * 55)
print()
delta_r2  = r2_2  - r2_1
delta_mse = mse1  - mse2
print(f'  R2 improvement from adding BMI : +{delta_r2:.3f}  ({delta_r2*100:.1f} percentage points)')
print(f'  MSE reduction from adding BMI  : -{delta_mse:.3f} mmHg2')
print()
print(f'  The multiple regression model explains {r2_2*100:.1f}% of SBP variance,')
print(f'  compared to {r2_1*100:.1f}% for the simple model.')

---
### Step 2.5 — Visualize: Actual vs. Predicted SBP

Since the multiple regression model lives in 3D space (Age × BMI × SBP), we cannot plot the regression plane directly in 2D. Instead, we use an **Actual vs. Predicted** plot:

- Each dot represents one test patient
- The x-axis is the **true** SBP
- The y-axis is the **predicted** SBP
- The **red dashed diagonal** is the line of perfect prediction (predicted = actual)

A better model has points **closer to the diagonal**. The more scattered the points, the larger the errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Unified axis limits for fair visual comparison
all_vals = np.concatenate([y1_test, y1_pred, y2_test, y2_pred])
lo = all_vals.min() - 3
hi = all_vals.max() + 3

for ax, yte, ypred, mse, r2, title, color in [
    (axes[0], y1_test, y1_pred, mse1, r2_1,
     f'Simple LR — Age only\nMSE={mse1:.2f}  R2={r2_1:.3f}', 'steelblue'),
    (axes[1], y2_test, y2_pred, mse2, r2_2,
     f'Multiple LR — Age + BMI\nMSE={mse2:.2f}  R2={r2_2:.3f}', 'darkorange'),
]:
    ax.scatter(yte, ypred, alpha=0.6, color=color,
               edgecolors='white', linewidth=0.3, s=42, zorder=2)
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.8, label='Perfect prediction', zorder=3)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel('Actual SBP (mmHg)', fontsize=11)
    ax.set_ylabel('Predicted SBP (mmHg)', fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_aspect('equal')

plt.suptitle('Task 2 — Actual vs. Predicted Systolic Blood Pressure', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
### Step 2.6 — Residual Analysis

We repeat the same residual checks from Task 1. With two predictors we also plot **residuals vs. each individual feature** to check that neither Age nor BMI has a systematic pattern in the residuals (which would indicate a non-linear relationship we are missing).

In [ ]:
residuals2 = y2_test - y2_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Plot 1: Residuals vs. Fitted ---
axes[0].scatter(y2_pred, residuals2, alpha=0.6, color='darkorange',
                edgecolors='white', linewidth=0.3, s=40)
axes[0].axhline(0, color='firebrick', linewidth=2, linestyle='--')
axes[0].axhspan(-rmse2, rmse2, alpha=0.07, color='darkorange')
axes[0].set_xlabel('Predicted SBP (mmHg)', fontsize=11)
axes[0].set_ylabel('Residual (mmHg)', fontsize=11)
axes[0].set_title('Residuals vs. Fitted', fontsize=12)

# --- Plot 2: Residuals vs. Age ---
axes[1].scatter(X2_test[:, 0], residuals2, alpha=0.6, color='steelblue',
                edgecolors='white', linewidth=0.3, s=40)
axes[1].axhline(0, color='firebrick', linewidth=2, linestyle='--')
axes[1].set_xlabel('Age (years)', fontsize=11)
axes[1].set_ylabel('Residual (mmHg)', fontsize=11)
axes[1].set_title('Residuals vs. Age', fontsize=12)

# --- Plot 3: Histogram of Residuals ---
mean_r2 = residuals2.mean()
axes[2].hist(residuals2, bins=18, color='darkorange', edgecolor='white', alpha=0.85)
axes[2].axvline(0,        color='firebrick', linewidth=2,   linestyle='--', label='Zero')
axes[2].axvline(mean_r2,  color='gold',      linewidth=1.5, linestyle='-',
                label=f'Mean = {mean_r2:.2f}')
axes[2].set_xlabel('Residual (mmHg)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title(f'Residual Distribution  (RMSE={rmse2:.2f})', fontsize=12)
axes[2].legend(fontsize=8)

plt.suptitle('Task 2 — Residual Analysis (Multiple Regression)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Residual summary (Multiple LR):')
print(f'  Mean   : {residuals2.mean():.4f}  (should be close to 0)')
print(f'  Std    : {residuals2.std():.4f} mmHg')
print(f'  Min    : {residuals2.min():.2f} mmHg')
print(f'  Max    : {residuals2.max():.2f} mmHg')

---
### Step 2.7 — Making Clinical Predictions

We now use the trained multiple regression model to screen **three new MRHA patients** who have just arrived at the clinic. For each patient, we predict their SBP and classify their hypertension risk stage.

In [ ]:
new_patients = pd.DataFrame({
    'Patient' : ['Patient A', 'Patient B', 'Patient C'],
    'age'     : [35,           58,           67         ],
    'bmi'     : [22.4,         30.1,         27.8       ]
})

X_new = new_patients[['age', 'bmi']].values
pred_sbp = model2.predict(X_new)

def classify_sbp(sbp):
    if sbp < 120:
        return 'Normal'
    elif sbp < 130:
        return 'Elevated'
    elif sbp < 140:
        return 'High Stage 1'
    else:
        return 'High Stage 2'

new_patients['Predicted SBP'] = np.round(pred_sbp, 1)
new_patients['Classification'] = [classify_sbp(s) for s in pred_sbp]

print('=== MRHA CardioWatch — Patient Risk Screening Results ===')
print()
print(new_patients.to_string(index=False))
print()
print(f'Model used: SBP = {b0_2:.2f} + {b1_2:.2f}*Age + {b2_2:.2f}*BMI')
print()
print('Manual calculation for Patient A (age=35, bmi=22.4):')
manual_a = b0_2 + b1_2 * 35 + b2_2 * 22.4
print(f'  SBP = {b0_2:.2f} + {b1_2:.2f}*35 + {b2_2:.2f}*22.4 = {manual_a:.2f} mmHg')

---
### Step 2.8 — Interactive Experiment with ipywidgets

This widget runs **both models** simultaneously so you can compare them side by side as you adjust the parameters.

**Experiments to try:**
1. Set noise to 2 mmHg — how close are the two models?  
2. Raise noise to 20 mmHg — does adding BMI still help?  
3. Try different seeds — observe how the R² gap between models varies with different splits  
4. Set test_size to 0.10 — notice how metrics become more volatile with a small test set

In [ ]:
def run_multiple_lr(test_size, noise_std, random_seed):
    np.random.seed(random_seed)
    n = 200
    age_w = np.random.uniform(25, 75, n)
    bmi_w = np.random.uniform(18.5, 38.0, n)
    sbp_w = 105 + 0.55 * age_w + 1.10 * bmi_w + np.random.normal(0, noise_std, n)
    sbp_w = np.clip(sbp_w, 95, 185)

    X1_w = age_w.reshape(-1, 1)
    X2_w = np.column_stack([age_w, bmi_w])
    y_w  = sbp_w

    X1tr, X1te, y_tr, y_te = train_test_split(
        X1_w, y_w, test_size=test_size, random_state=random_seed
    )
    X2tr, X2te, _,    _    = train_test_split(
        X2_w, y_w, test_size=test_size, random_state=random_seed
    )

    m1 = LinearRegression().fit(X1tr, y_tr)
    m2 = LinearRegression().fit(X2tr, y_tr)

    yp1 = m1.predict(X1te)
    yp2 = m2.predict(X2te)

    mse_1, r2_w1 = mean_squared_error(y_te, yp1), r2_score(y_te, yp1)
    mse_2, r2_w2 = mean_squared_error(y_te, yp2), r2_score(y_te, yp2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Model Comparison  |  test_size={test_size:.0%}  noise={noise_std} mmHg  seed={random_seed}',
        fontsize=12
    )

    lo = min(y_te.min(), yp1.min(), yp2.min()) - 3
    hi = max(y_te.max(), yp1.max(), yp2.max()) + 3

    for ax, ypred, mse_v, r2_v, title, color in [
        (axes[0], yp1, mse_1, r2_w1,
         f'Simple LR (Age only)\nMSE={mse_1:.2f}  R2={r2_w1:.3f}', 'steelblue'),
        (axes[1], yp2, mse_2, r2_w2,
         f'Multiple LR (Age + BMI)\nMSE={mse_2:.2f}  R2={r2_w2:.3f}', 'darkorange'),
    ]:
        ax.scatter(y_te, ypred, alpha=0.6, color=color,
                   edgecolors='white', linewidth=0.3, s=38, zorder=2)
        ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.8, label='Perfect prediction', zorder=3)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_xlabel('Actual SBP (mmHg)', fontsize=11)
        ax.set_ylabel('Predicted SBP (mmHg)', fontsize=11)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    delta = r2_w2 - r2_w1
    print(f'  Simple LR    MSE={mse_1:.3f}  RMSE={np.sqrt(mse_1):.3f}  R2={r2_w1:.3f}')
    print(f'  Multiple LR  MSE={mse_2:.3f}  RMSE={np.sqrt(mse_2):.3f}  R2={r2_w2:.3f}')
    print(f'  R2 improvement by adding BMI: +{delta:.3f}  ({delta*100:.1f} pp)')


widgets.interact(
    run_multiple_lr,
    test_size=widgets.FloatSlider(
        min=0.10, max=0.40, step=0.05, value=0.20,
        description='Test Size:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    ),
    noise_std=widgets.FloatSlider(
        min=2.0, max=25.0, step=1.0, value=6.0,
        description='Noise Level (sigma):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    ),
    random_seed=widgets.IntSlider(
        min=0, max=50, step=1, value=7,
        description='Random Seed:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%')
    )
)

---
---
## Conclusions

This notebook demonstrated the full linear regression pipeline on a realistic health dataset:

| | Task 1 | Task 2 |
|-|--------|--------|
| **Model** | Simple Linear Regression | Multiple Linear Regression |
| **Predictors** | Age | Age + BMI |
| **Equation** | SBP = β₀ + β₁·Age | SBP = β₀ + β₁·Age + β₂·BMI |
| **Key metric** | R² ≈ 0.56 | R² ≈ 0.73 |

Adding BMI as a second predictor increased R² by approximately 17 percentage points, demonstrating that **multiple regression captures more of the variance in the target** when the added predictor is genuinely informative and not redundant (low multicollinearity with existing predictors).

---
## Reflection Questions

Write your answers in the Markdown cells below each question.

---

**Q1.** Interpret the slope $\beta_1$ from Task 1 in one sentence suitable for a medical report — no math, plain language only.

*Answer:* ...

---

**Q2.** In Task 2, the coefficient for Age changed slightly compared to Task 1. Why does adding BMI change the Age coefficient?

*Answer:* ...

---

**Q3.** Based on the residual plots, which assumptions of linear regression appear to hold? Which might be violated?

*Answer:* ...

---

**Q4.** Using the interactive widget (Task 2), describe what happens to R² as you increase the noise level from 2 to 20 mmHg. Explain *why* this happens.

*Answer:* ...

---

**Q5.** What additional patient variables could improve the model further? Name at least two and explain the clinical rationale.

*Answer:* ...

---

**Q6. (Ethics)** The MRHA wants to use the multiple regression model to automatically flag patients for mandatory medication without physician review. What are the risks of this approach?

*Answer:* ...